In [ ]:
#Libraries
import requests
import pandas as pd
import seaborn as sns
import datetime
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from imblearn.pipeline import Pipeline
from scipy.stats.mstats import winsorize
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import classification_report,confusion_matrix, ConfusionMatrixDisplay, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# United States Airlines

## Data Science

In [ ]:
#Load Datasets
airlines_data = pd.read_excel('Airlines.xlsx')
airports_data = pd.read_excel('airports.xlsx')[['ident','type','elevation_ft','iata_code','scheduled_service']]
runways_data = pd.read_excel('runways.xlsx')[['airport_ident', 'length_ft', 'width_ft', 'surface', 'lighted', 'closed']]

In [ ]:
#Airlines dataset
airlines = airlines_data.copy() # for faster data reloading
airlines

In [ ]:
#Check Null Values
airlines.isna().sum() 

No Null values

In [ ]:
#Airports Dataset
#Keeping only necessary columns
airports = airports_data.copy() # for faster data reloading
airports

In [ ]:
#check missing values
airports.isna().sum() 

Missing iata_code values should be dropped as it is the key that will join with airlines table columns airportfrom and airportto.

In [ ]:
#Drop Iata_code missing values
airports = airports.dropna(subset = ['iata_code'])
airports.isna().sum() 

In [ ]:
#Runways dataset
#Keeping only necessary columns
runways = runways_data.copy() # for faster data reloading
runways

In [ ]:
#check missing values
runways.isna().sum()

Only the columns related to operations, infrastructure and time were retained for this anlaysis as they directly influence the delay. Other discriptive and textual columns have been excluded to reduce noise and dimensionality.

### Exctracting the data from Webpage

In [ ]:
# Necessary to comply with robot.html regulations
headers = {
    'User-Agent': (
        'MyResearchBot/1.0 (https://example.com/contact) '
        'Requests/BeautifulSoup; respectful bot following Wikimedia rules'
    )
}
# Extracting airlines information
url = 'https://en.wikipedia.org/wiki/List_of_airlines_of_the_United_States'
response1 = requests.get(url, headers = headers)
tables = pd.read_html(response1.text)  # pandas table web scrapping tool

In [ ]:
#scrapping all tables with necessary columns
mainline = tables[0][['IATA', 'Founded']]
regional = tables[1][['IATA', 'Founded']]
commuter = tables[2][['IATA', 'Founded']]
charter = tables[3][['IATA', 'Founded']]
cargo = tables[4][['IATA', 'Founded']]
# merge all airline tables
table_list = [mainline, regional, commuter, charter, cargo] 
airlines_wiki = pd.concat(table_list, ignore_index = True) 
airlines_wiki

In [ ]:
#Drop Null Values
airlines_wiki = airlines_wiki.dropna(subset=['IATA'])
airlines_wiki = airlines_wiki.dropna(subset=['Founded'])

In [ ]:
#Get the years of experience
airlines_wiki['Years of Experience'] = datetime.date.today().year - airlines_wiki['Founded']
airlines_wiki = airlines_wiki.drop(columns = ['Founded'])
airlines_wiki

In [ ]:
#Scrapping Busiest airports webpage
url2 = 'https://en.wikipedia.org/wiki/List_of_the_busiest_airports_in_the_United_States'
response2 = requests.get(url2, headers = headers)
tables2 = pd.read_html(response2.text)  # pandas table web scrapping tool

In [ ]:
# Extract Large Airports hub table
large_airports = tables2[0][['IATA Code', '2024[2]', '2023[3]']]
#Label hubs as large
large_airports['Hub Size'] = 'Large'
#Get total of passenger traffic of last two years
large_airports['Total Traffic'] = large_airports['2024[2]'] + large_airports['2023[3]']
large_airports = large_airports.drop(['2024[2]', '2023[3]'], axis=1)
large_airports

In [ ]:
# Extract Medium Airports hub table
medium_airports = tables2[1][['IATA Code', '2024[2]', '2023[3]']]
#Label hubs as medium
medium_airports['Hub Size'] = 'Medium'
#Get total of passenger traffic of last two years
medium_airports['Total Traffic'] = medium_airports['2024[2]'] + medium_airports['2023[3]']
medium_airports = medium_airports.drop(['2024[2]', '2023[3]'], axis=1)
medium_airports

Since the its not specified in the brief, I aggregated 2023 and 2024 passenger emplanements to create a recent and stable proxy for airport congestion.

In [ ]:
#Join the airport tables
table_list2 = [large_airports, medium_airports] 
airports_wiki = pd.concat(table_list2, ignore_index = True) 
airports_wiki

### Joining all the data

In [ ]:
# Merge airlines table with airports table
#first merge airport_from to iata_code on data
airports_from = airports.add_prefix('from_') # add from_ prefix to columns

data = airlines.merge(
    airports_from,
    how ='left', 
    left_on = 'AirportFrom',
    right_on = 'from_iata_code'
)
data

In [ ]:
#second merge airport_to to iata_code on data
airports_to = airports.add_prefix('to_') # add to prefix to columns

data = data.merge(
    airports_to,
    how ='left', 
    left_on = 'AirportTo',
    right_on = 'to_iata_code'
)
data

In [ ]:
#Next Merge the merged data with runways table
#first we aggregate the runways table to avoid data duplication
runways_agg = (
    runways.groupby('airport_ident').agg(
        avg_runway_len = ('length_ft', 'mean'),
        avg_runway_width = ('width_ft', 'mean'),
        lighted = ('lighted', 'mean'),
        closed = ('closed', 'mean')
    ).reset_index()
)
runways_agg

In [ ]:
#Next we merge the tables
#first the from_ident with airport ident
runways_from = runways_agg.add_prefix('from_')

data = data.merge(
    runways_from,
    how = 'left',
    left_on = 'from_ident',
    right_on = 'from_airport_ident'
)
data

In [ ]:
#second the to_ident with airport ident
runways_to = runways_agg.add_prefix('to_')

data = data.merge(
    runways_to,
    how = 'left',
    left_on = 'to_ident',
    right_on = 'to_airport_ident'
)
data

In [ ]:
#Next merge the data with the Airlines wiki page
data = data.merge(
    airlines_wiki,
    how = 'left',
    left_on = 'Airline',
    right_on = 'IATA'
)
data

In [ ]:
# Drop unnecessary columns
data = data.drop(columns = ['IATA'])
data

In [ ]:
#Next merge the data with airports wiki page table
#Merge using aiport from
airports_wiki_from = airports_wiki.add_prefix('from_')


data = data.merge(
    airports_wiki_from,
    how = 'left',
    left_on = 'AirportFrom',
    right_on = 'from_IATA Code'
)
data

In [ ]:
#Merge using aiport to
airports_wiki_to = airports_wiki.add_prefix('to_')


data = data.merge(
    airports_wiki_to,
    how = 'left',
    left_on = 'AirportTo',
    right_on = 'to_IATA Code'
)
data

### Cleaning the data

In [ ]:
#drop further unnecessary columns
data = data.drop(columns = ['id', 'from_iata_code', 'to_iata_code', 'from_IATA Code', 'to_IATA Code'])
data

In [ ]:
#check missing values by column
data.isna().sum()

In [ ]:
#replace missing values with Small in Hub Size columns
cols = ['from_Hub Size', 'to_Hub Size']
data[cols] = data[cols].fillna('Small')
data

For this case since these missing values aren't either a large or medium hub we could estimate that they are small hubs, and so we can replace all the missing values with 'Small'.

In [ ]:
# Categorical columns from airport
data['from_type'] = data['from_type'].fillna(data['from_type'].mode()[0])
data['to_type'] = data['to_type'].fillna(data['to_type'].mode()[0])
data['to_scheduled_service'] = data['to_scheduled_service'].fillna(data['to_scheduled_service'].mode()[0])
data['from_scheduled_service'] = data['from_scheduled_service'].fillna(data['from_scheduled_service'].mode()[0])
data['from_lighted'] = data['from_lighted'].fillna(data['from_lighted'].mode()[0])
data['to_lighted'] = data['to_lighted'].fillna(data['to_lighted'].mode()[0])
data['from_closed'] = data['from_closed'].fillna(data['from_closed'].mode()[0])
data['to_closed'] = data['to_closed'].fillna(data['to_closed'].mode()[0])

These categorical columns were filled with the mode as they represent discrete classes and most airports will share common categories

In [ ]:
data.isna().sum()

In [ ]:
#Check the elevation_ft columns
print(data['from_elevation_ft'].describe(),'\n')
print(data['to_elevation_ft'].describe())

In [ ]:
plt.hist(data['from_elevation_ft'])
plt.title('Source Elavation')
plt.xlabel('Elevation (ft)')
plt.show()

In [ ]:
plt.hist(data['to_elevation_ft'])
plt.title('Destination Elevation')
plt.xlabel('Elevation (ft)')
plt.show()

In [ ]:
sns.boxplot(data=data, y = 'from_elevation_ft', x = 'Delay')
plt.title('Source Elavation')
plt.ylabel('Elevation (ft)')
plt.xlabel("Delay (0 = On time, 1 = Delayed)")
plt.show()

In [ ]:
sns.boxplot(data=data, y = 'to_elevation_ft', x = 'Delay')
plt.xlabel("Delay (0 = On time, 1 = Delayed)")
plt.ylabel('Elevation (ft)')
plt.title('Destination Elevation')
plt.show()

From the above graphs we can see that the elevations are completely identical whether it is source or destination, or if the flights are delayed or not. The distribution is extremely positively skewed, which should be expected as some airports are known to go up to extreme elevations. And from looking at the boxplot we can see such outliers go well over the extreme max line value of just over 2000 ft, and reaching around 9000 ft. So, these values should be retained but capped, and missing values should be filled with the median as this preserves central tendency. Since this data includes negative values, it is not advisable to log transform it to normalise the distribution. 

For the rest of the visualizations I will only use the _from columns as all values on the _to columns show identical data.

In [ ]:
#replace missing values first
data['from_elevation_ft'] = data['from_elevation_ft'].fillna(data['from_elevation_ft'].median())
data['to_elevation_ft'] = data['to_elevation_ft'].fillna(data['to_elevation_ft'].median())

In [ ]:
#Cap values
data['from_elevation_ft'] = winsorize(data['from_elevation_ft'], limits = [0.01, 0.01])
data['to_elevation_ft'] = winsorize(data['to_elevation_ft'], limits = [0.01, 0.01])
data.isna().sum()

In [ ]:
plt.hist(data['from_elevation_ft'])
plt.title('Source Elavation')
plt.xlabel('Elevation (ft)')
plt.show()

In [ ]:
sns.boxplot(data=data, y = 'to_elevation_ft', x = 'Delay')
plt.xlabel('Delay (0 = On time, 1 = Delayed'))
plt.ylabel('Elevation (ft)')
plt.title('Destination Elevation')
plt.show()

In [ ]:
#Checking the average runway lengths
data['from_avg_runway_len'].describe()

In [ ]:
plt.hist(data['from_avg_runway_len'])
plt.title('Source Runway Length')
plt.xlabel('Runway Length(ft)')
plt.show()

In [ ]:
sns.boxplot(data=data, y = 'from_avg_runway_len', x = 'Delay')
plt.xlabel("Delay (0 = On time, 1 = Delayed)")
plt.ylabel('Runway Length(ft)')
plt.title('Source Runway Length')
plt.show()

The graphs above shows a moderately normal distribuition with a slight negative skew, and the mean being very close to the median. As for outliers there is very little to see, with just a small group being bellow the minimum line. So we can just replace the null values with the median to preserve central tendency.

In [ ]:
data['from_avg_runway_len'] = data['from_avg_runway_len'].fillna(data['from_avg_runway_len'].median())
data['to_avg_runway_len'] = data['to_avg_runway_len'].fillna(data['to_avg_runway_len'].median())
data.isna().sum()

In [ ]:
#Checking the average runway widths
data['from_avg_runway_width'].describe()

In [ ]:
plt.hist(data['from_avg_runway_width'])
plt.title('Source Runway Width')
plt.xlabel('Runway Width(ft)')
plt.show()

In [ ]:
sns.boxplot(data=data, y = 'from_avg_runway_width', x = 'Delay')
plt.xlabel("Delay (0 = On time, 1 = Delayed)")
plt.ylabel('Runway Widht(ft)')
plt.title('Source Runway Width')
plt.show()

The Graphs show very unequal distribution with negative skewness, and shows a few outliers with an obvious extreme outlier at around just over 800 ft. Since runways can genuinely be very large at major airports and very small in smaller airports, this is normal, so for this case most values shall be retained and only the very large values will be winsorized to prevent distortion. 

In [ ]:
#Replace missing values
data['from_avg_runway_width'] = data['from_avg_runway_width'].fillna(data['from_avg_runway_width'].mode()[0])
data['to_avg_runway_width'] = data['to_avg_runway_width'].fillna(data['to_avg_runway_width'].mode()[0])

In [ ]:
#Removing Extreme outliers at 1% and 99%
data['from_avg_runway_width'] = winsorize(data['from_avg_runway_width'], limits = [0.01, 0.01])
data['to_avg_runway_width'] = winsorize(data['to_avg_runway_width'], limits = [0.01, 0.01])

In [ ]:
plt.hist(data['from_avg_runway_width'])
plt.title('Source Runway Width')
plt.xlabel('Runway Width(ft)')
plt.show()

In [ ]:
sns.boxplot(data=data, y = 'from_avg_runway_width', x = 'Delay')
plt.xlabel("Delay (0 = On time, 1 = Delayed)")
plt.ylabel('Runway Widht(ft)')
plt.title('Source Runway Width')
plt.show()

In [ ]:
data.isna().sum()

For the Years of Experience column, we can fill the missing values with the median but can also add an indicator that these values are missing by creating a missing value column.

In [ ]:
#Fill Missing values for Years of Experience
#Create Missing values indicator column
data['Missing Experience'] = data['Years of Experience'].isna().astype(int)
data['Years of Experience'] = data['Years of Experience'].fillna(data['Years of Experience'].median())
data.isna().sum()

For the Total traffic columns we can replace the missing values with 0, as this means that the airport is not listed amongst the busiest airports.

In [ ]:
#Fill total trafic missing values with 0
#Create Missing values indicator column
data['Missing from_Total Traffic'] = data['from_Total Traffic'].isna().astype(int)
data['Missing to_Total Traffic'] = data['to_Total Traffic'].isna().astype(int)

data["from_Total Traffic"] = data["from_Total Traffic"].fillna(0)
data["to_Total Traffic"] = data["to_Total Traffic"].fillna(0)
data.isna().sum()

All Missing Values were dealt with, so we can now move on to the data science analysis. The ident columns will be used for hypothesis testing and then later dropped.

## 3. Data Visualization

In [ ]:
#check unique airline codes
data['Airline'].unique()

According to Google the IATA code WN represents the southwestern airlines.

In [ ]:
#Southwestern Airlines
southwestern = data[data['Airline'] == 'WN']
print(f'{len(southwestern)} of all airlines are Southwestern.')

In [ ]:
#plot flight delays
sw_delay = southwestern['Delay'].value_counts().sort_index()
sw_delay.plot(kind="bar")
plt.xlabel("Delay (0 = On time, 1 = Delayed)")
plt.ylabel("Number of Flights")
plt.title("Delay")
plt.show()
print(f'About {sw_delay[0]/len(southwestern)*100:.2f}% of all flights were on time, and {sw_delay[1]/len(southwestern)*100:.2f}% were delayed.')

In [ ]:
#Keep only delayed flights
delayed_flights = data[data['Delay']==1]
delayed_flights.head(5)

In [ ]:
#Plot flight delay comparison
airlines_delay = delayed_flights['Airline'].value_counts().sort_index()
airlines_delay.plot(kind="bar")
plt.ylabel('Number of Delayed Flights')
plt.title('Delay')
plt.show()
print(airlines_delay)

Compared to other flights, southwestern airlines show the most delayed flights by quite a distance with the second closest being Delta Airlines(DL) with 27452 delayed flights, and the lowest being Alpha Sky(AS) with 3892 delayed flights.

In [ ]:
#b Safest Weekdays to travel
hue_order = sorted(data['Delay'].unique(), reverse=True) # reverse delay values

sns.histplot(
    data=data,
    x='DayOfWeek',
    hue='Delay',
    multiple='fill',
    discrete=True,
    hue_order=hue_order
)

plt.ylabel('Number of Flights')
plt.xlabel('Day of week')
plt.title('Best Day of the Week')
plt.show()

weekday_delay = (
    data
    .groupby(['DayOfWeek', 'Delay'])
    .size()
    .groupby(level=0)
    .apply(lambda x: 100 * x / x.sum())
    .unstack()
    .round(2)
)

print(weekday_delay)

The safest day of the week to travel is Friday(6) with about 60% of flights being on time.

In [ ]:
# c.Best Airlines for each duration
#Find a value to seperate into 3 sections
data['Length'].max()/3

In [ ]:
#Create distance labels
data['Flight Duration'] = pd.cut(data['Length'],labels =['Short Distance', 'Medium Distance', 'Long Distance'],
                                bins = [0, 218, 436, 655], include_lowest = True)
data

In [ ]:
# Plot shorth flight distance by airline
short_distance = data[data['Flight Duration'] == 'Short Distance']

hue_order = sorted(short_distance["Delay"].unique(), reverse=True)

sns.histplot(
    data=short_distance,
    x='Airline',
    hue='Delay',
    multiple='fill',
    discrete=True,
    hue_order=hue_order
)

plt.ylabel('Number of Flights')
plt.xlabel('Airline')
plt.title('Best Airline for Short Distance Flights')
plt.show()

weekday_delay = (
    short_distance
    .groupby(['Airline", "Delay'])
    .size()
    .groupby(level=0)
    .apply(lambda x: 100 * x / x.sum())
    .unstack()
    .round(2)
)

print(weekday_delay)

For short distance flights, the best Airlines are: 1.Mesa Airlines(YV) with 75% on time flights, 2.PSA Airlines(OH) with 72%, and 3.United Airlines(UA) with 70%.

In [ ]:
medium_distance = data[data['Flight Duration'] == 'Medium Distance']

hue_order = sorted(medium_distance["Delay"].unique(), reverse=True)

sns.histplot(
    data=medium_distance,
    x="Airline",
    hue="Delay",
    multiple="fill",
    discrete=True,
    hue_order=hue_order
)

plt.ylabel("Number of Flights")
plt.xlabel("Airline")
plt.title("Best Airline for Medium Distance Flights")
plt.show()

weekday_delay = (
    medium_distance
    .groupby(["Airline", "Delay"])
    .size()
    .groupby(level=0)
    .apply(lambda x: 100 * x / x.sum())
    .unstack()
    .round(2)
)

print(weekday_delay)

For Medium distance flights, the best Airlines are 1.Mesa Airlines(YV) with 74% on time flights, 2.Envoy Air(MQ) with 73% and 
3.Alpha Sky(AS) with 62%.

In [ ]:
long_distance = data[data['Flight Duration'] == 'Long Distance']

hue_order = sorted(long_distance["Delay"].unique(), reverse=True)

sns.histplot(
    data=long_distance,
    x="Airline",
    hue="Delay",
    multiple="fill",
    discrete=True,
    hue_order=hue_order
)

plt.ylabel("Number of Flights")
plt.xlabel("Airline")
plt.title("Best Airline for Long Distance Flights")
plt.show()

weekday_delay = (
    long_distance
    .groupby(["Airline", "Delay"])
    .size()
    .groupby(level=0)
    .apply(lambda x: 100 * x / x.sum())
    .unstack()
    .round(2)
)

print(weekday_delay)

For long distance flights, the best Airlines are 1United Airlines(UA) with 60% on time flights and 2.Delta Air Lines(DL) with 51%.

In [ ]:
#d. Pattern in long distance flight departure times.
#leave only Long distance rows
x = data.copy()
long_distance = x[x['Flight Duration'] == 'Long Distance']
#convert time column to hour

long_distance['Departure Hour'] = x['Time'] // 60
long_distance[['Departure Hour']]

In [ ]:
#Plot time distribution
plt.hist(long_distance['Departure Hour'])
plt.xlabel('Departure Hour')
plt.ylabel('Number of Flights')
plt.title('Departure Times- Long Distance Flights')
plt.show()

Long distance flights appear to be mostly during the morning times between the hours of 9 to 11 am, with a considerable number coming also at around 5pm, meaning airlines prioritize early morning flights for long distance flights.

In [ ]:
#4. Delayed flights Large vs Medium hubs for source airport
large_medium = data[data['from_Hub Size']!='Small']

delays = large_medium.groupby('from_Hub Size')['Delay'].mean()
delays

In [ ]:
#plot distribution
delays.plot(kind='bar')
plt.ylabel('Delay Rate')
plt.xlabel('Hub Size')
plt.title('Delay Rate - Large vs Medium Hubs')

Source Medium hubs have a slightly higher chance of delay of 49% compared to medium hubs with 47% chance of delay.

In [ ]:
#. Delayed flights Large vs Medium hubs for destination airport
large_medium = data[data['to_Hub Size']!='Small']

delays = large_medium.groupby('to_Hub Size')['Delay'].mean()
delays

In [ ]:
#plot distribution
delays.plot(kind='bar')
plt.ylabel('Delay Rate')
plt.xlabel('Hub Size')
plt.title('Delay Rate - Large vs Medium Hubs')

Destination Medium hubs have a higher chance of delay of 52% compared to medium hubs with 42% chance of delay.

### 5.Hypothesis Testing

In [ ]:
#5.a
#h0 - Airport altitude has impact on flight delays for incoming and departing flights
#h1 - Airtport altitude do not impact flight delays
#Source Airport
delay = data[data['Delay']==1]['from_elevation_ft']
non_delay = data[data['Delay']==0]['from_elevation_ft']

t_stat, p_value = ttest_ind(delay, non_delay, equal_var = False)
t_stat, p_value

In [ ]:
#Destination Airport
delay = data[data['Delay']==1]['to_elevation_ft']
non_delay = data[data['Delay']==0]['to_elevation_ft']

t_stat, p_value = ttest_ind(delay, non_delay, equal_var = False)
t_stat, p_value

For this study we can confidently reject the null hypothesis and go for the alternative which is that airport altitude for either origin or destination does not impact flight delays with both p-values being far smaller than 0.05.

In [ ]:
#b.
#h0 - The number of runways at an airport affects flight delays
#h1 -The number of runways at an airport does not impact flight delays
#Source Airport

data_runway = data.copy()

#get runway count
runway_count = (runways.groupby('airport_ident').size().reset_index(name = 'runway_count'))
data_runway = data_runway.merge(
    runway_count,
    how = 'left',
    left_on ='from_airport_ident',
    right_on = 'airport_ident'
)
data_runway.rename(columns ={'runway_count': 'from_runway_count'}, inplace = True)
data_runway.drop(columns=['airport_ident'], inplace = True)

#fill missing values with 1 as all aiports have at least one
data_runway['from_runway_count'] = data_runway['from_runway_count'].fillna(1)

delay = data_runway[data_runway['Delay']==1]['from_runway_count']
non_delay = data_runway[data_runway['Delay']==0]['from_runway_count']

#perfrom test
t_stat, p_value = ttest_ind(delay, non_delay, equal_var=False)
t_stat, p_value

In [ ]:
#Destination Airport
#get runway count
data_runway = data_runway.merge(
    runway_count,
    how = 'left',
    left_on ='to_airport_ident',
    right_on = 'airport_ident'
)
data_runway.rename(columns ={'runway_count': 'to_runway_count'}, inplace = True)
data_runway.drop(columns=['airport_ident'], inplace = True)

#fill missing values with 1 as all aiports have at least one
data_runway['to_runway_count'] = data_runway['to_runway_count'].fillna(1)

delay = data_runway[data_runway['Delay']==1]['to_runway_count']
non_delay = data_runway[data_runway['Delay']==0]['to_runway_count']

#perfrom test
t_stat, p_value = ttest_ind(delay, non_delay, equal_var=False)
t_stat, p_value

For this study we can confidently reject the null hypothesis and go for the alternative which is that number of runways for either origin or destination does not impact flight delays with both p-values being far smaller than 0.05.

In [ ]:
#5.c
#h0- The duration of a flight affects flight delays
#h1 - The duration of a flights does not impact delay

delay = data[data['Delay']==1]["Length"]
non_delay = data[data['Delay']==0]["Length"]

t_stat, p_value = ttest_ind(delay, non_delay, equal_var = False)
t_stat, p_value

For this case the p value is a lot smaller than 0.05 so we can confidently reject the null hypothesis and conclude that flight duration does not impact delay.

In [ ]:
data = data.drop(columns = ['from_ident', 'to_ident', 'from_airport_ident', 'to_airport_ident'])

### 6.Correlation

In [ ]:
#.Plot Correlation Matrix
# Calculate correlation matrix for numeric columns
correlation_matrix = data.select_dtypes(include='number').corr()

#plot heatmap
plt.figure(figsize = (20,10))
sns.heatmap(correlation_matrix, annot = True, fmt ='.2f',cmap='coolwarm')
plt.title('Heatmap of Correlation Matrix')
plt.show()

In this Dataset there does not seem to be much correlation between the features, there are features that show some correlation such as from average runway length showig just over 55% correlation with from average runway width and to total traffic, with the rest of the features not even reaching 50% correlation and a considerable number of features showing negative correlation.

## Machine Learning

In [ ]:
#plot the distribuition
delay_counts = data["Delay"].value_counts().sort_index()

delay_counts.plot(kind="bar")
plt.xlabel("Delay (0 = On time, 1 = Delayed)")
plt.ylabel("Number of Flights")
plt.title("Delay")
plt.show()
print(f'About {delay_counts[0]/len(data)*100:.2f}% of all flights were on time, and {delay_counts[1]/len(data)*100:.2f}% were delayed.')

In [ ]:
data.info()

In [ ]:
#Categorical columns
cat =['Airline', 'Flight', 'DayOfWeek','AirportFrom', 'AirportTo','from_type', 'to_type', 'from_scheduled_service', 'to_scheduled_service', 
            'from_lighted','to_lighted', 'from_closed', 'to_closed' ,'from_Hub Size', 'to_Hub Size', 'Flight Duration', 'Missing Experience']
cat

In [ ]:
#numerical columns
cats = cat.copy()
cats.append('Delay')
cats
cols = data.columns
num = [col for col in cols if col not in cats]
num

In [ ]:
#1.Sepperate data and encode categorical columns
data1 = data.copy()

X = data1.drop('Delay', axis = 1) #drop target column
y = data1['Delay'] #target column

In [ ]:
#2.a. Train Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

#### Logistic Regression using Stochastic Gradient Descent

In [ ]:
#2.b.c. Unified preprocessing for both numeric and categorical data 
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore', drop ='first')
preprocesssing_sgdlr = ColumnTransformer([
    ('num', scaler, num),
    ('cat',encoder, cat)
])
preprocesssing_sgdlr

In [ ]:
#fit Logistic regression model into pipeline
sgdlr = SGDClassifier(loss = 'log_loss', max_iter = 10000) #loss = log_loss for logistic regression
pipeline_sgdlr = Pipeline([
    ('prep', preprocesssing_sgdlr),
    ('model', sgdlr)
])
pipeline_sgdlr

In [ ]:
#train model
pipeline_sgdlr.fit(X_train, y_train)
y_train_pred_sgdlr = pipeline_sgdlr.predict(X_train)
y_test_pred_sgdlr= pipeline_sgdlr.predict(X_test)

In [ ]:
# accuracy score and classification Report
train_acc_sgdlr = accuracy_score(y_train, y_train_pred_sgdlr)
test_acc_sgdlr = accuracy_score(y_test, y_test_pred_sgdlr)
print('\t\t\tLogistic Regression:\n')
print('\t\tTrain Classification Report:')
print(classification_report(y_train, y_train_pred_sgdlr),'\n')
print('\t\tTest Classification Report:')
print(classification_report(y_test, y_test_pred_sgdlr),'\n')
print(f'Train: {train_acc_sgdlr}')
print(f'Test: {test_acc_sgdlr}')

For Logistic Regression using SGD classifier, the model gives an accuracy score of 65% and 64% for train and test data respectively, meaning the model accounts well for overfitting and generalizes well but does not perform as well as needed with only moderate accuracy. This could mean that there are other external factors at play, or this model is not the best for this case study.

#### Decision Tree

In [ ]:
#2.b.c.  Unified preprocessing for both numeric and categorical data 
encoder = OneHotEncoder(handle_unknown='ignore')
preprocesssing_dt = ColumnTransformer([
    ('num', scaler, num),
    ('cat',encoder, cat)
])
preprocesssing_dt

In [ ]:
#fit Decision Tree model into pipeline
dt = DecisionTreeClassifier(random_state = 42)
pipeline_dt = Pipeline([
    ('prep', preprocesssing_dt),
    ('model', dt)
])
pipeline_dt

In [ ]:
#train model
pipeline_dt.fit(X_train, y_train)
y_train_pred_dt = pipeline_dt.predict(X_train)
y_test_pred_dt= pipeline_dt.predict(X_test)

In [ ]:
# accuracy score and classification Report

train_acc_dt = accuracy_score(y_train, y_train_pred_dt)
test_acc_dt = accuracy_score(y_test, y_test_pred_dt)
print('\t\t\tDecision Tree:\n')
print('\t\tTrain Classification Report:')
print(classification_report(y_train, y_train_pred_dt),'\n')
print('\t\tTest Classification Report:')
print(classification_report(y_test, y_test_pred_dt),'\n')
print('Accuracy Score:')
print(f'Train: {train_acc_dt}')
print(f'Test: {test_acc_dt}')

Without accounting for overfitting Decision Tree model performs well on train data with an accuracy score of 84% but does not perform as well on test data, giving an accuracy score of only 61%, meaning that hyperparameter tuning should be applied to deal with the overfitting of the model. The model also takes a long time to train without any parameters, specially without max depth.

### Hyperparameter Tuning

In [ ]:
#fit Decision Tree model into pipeline with new params
dt_params = DecisionTreeClassifier(random_state = 42, max_depth = 11, ccp_alpha = 0.0, min_samples_split =10)
pipeline_dt_params = Pipeline([
    ('prep', preprocesssing_dt),
    ('model', dt_params)
])
pipeline_dt_params

In [ ]:
#train model
pipeline_dt_params.fit(X_train, y_train)
y_train_pred_dt_params = pipeline_dt_params.predict(X_train)
y_test_pred_dt_params = pipeline_dt_params.predict(X_test)

In [ ]:
# accuracy score and classification Report

train_acc_dt_params = accuracy_score(y_train, y_train_pred_dt_params)
test_acc_dt_params = accuracy_score(y_test, y_test_pred_dt_params)
print('\t\t\tDecision Tree:\n')
print('\t\tTrain Classification Report:')
print(classification_report(y_train, y_train_pred_dt_params),'\n')
print('\t\tTest Classification Report:')
print(classification_report(y_test, y_test_pred_dt_params),'\n')
print('Accuracy Score:')
print(f'Train: {train_acc_dt_params}')
print(f'Test: {test_acc_dt_params}')

After performing Hyperparameter Tuning on the Tree Decision Classifier to account for overfitting the best result the model can achieve is 66% for train and 65% for testing. Gridsearch was attempted for best parameters to use but due to the large dataset and number of features this was not possible as it would take too long, so this result was achieved by manually tuning. Again, it could be that there are features that are not in the data such as the weather which is probably the biggest influence for flight delay and ultimately, flight cancelation.

### Model Voting using Stratified 5-Fold Cross validation

In [ ]:
#fit Logistic Regression model into pipeline
lr = LogisticRegression( max_iter = 10000) #changed to normal LR model as SGDC does not work with hard voting
pipeline_lr = Pipeline([
    ('prep', preprocesssing_lr),
    ('model', lr)
])
pipeline_lr

In [ ]:
#train model
pipeline_lr.fit(X_train, y_train)
y_train_pred_lr = pipeline_lr.predict(X_train)
y_test_pred_lr= pipeline_lr.predict(X_test)

In [ ]:
#initialize estimator models
estimators = [
    ('log-reg', pipeline_lr),
    ('dtc', pipeline_dt_params)
]

In [ ]:
#create hard voting 
ensemble = VotingClassifier(estimators, voting = 'hard')

#setup kfold cross validation
kfold = KFold(n_splits=5, shuffle= True, random_state= 42)

#Evaluate models using corss validation
results = cross_val_score(ensemble, X_train, y_train, cv=kfold)

#fit ensemble
ensemble.fit(X_train, y_train)
test_accuracy = ensemble.score(X_test, y_test)
y_pred_hard = ensemble.predict(X_test)

#print test accuracy and classification report
print("Hard Voting - Classification Report")
print(classification_report(y_test, y_pred_hard))
print(f'Test accuracy of the ensemble model: {test_accuracy:.2f}')

### 3. Gradient Boosting

In [ ]:
#fit Gradient Bossting model into pipeline
gb = GradientBoostingClassifier(n_estimators=100, random_state=7)
pipeline_gb = Pipeline([
    ('prep', preprocesssing_dt),
    ('model', gb)
])
pipeline_gb

In [ ]:
#train model
pipeline_gb.fit(X_train, y_train)
y_train_pred_gb= pipeline_gb.predict(X_train)
y_test_pred_gb= pipeline_gb.predict(X_test)

In [ ]:
# accuracy score and classification Report
train_acc_gb = accuracy_score(y_train, y_train_pred_gb)
test_acc_gb = accuracy_score(y_test, y_test_pred_gb)
print('\t\t\tGradient Boosting:\n')
print('\t\tTrain Classification Report:')
print(classification_report(y_train, y_train_pred_gb),'\n')
print('\t\tTest Classification Report:')
print(classification_report(y_test, y_test_pred_gb),'\n')
print('Accuracy Score:')
print(f'Train: {train_acc_gb}')
print(f'Test: {test_acc_gb}')

### 3. Comparing the models

In [ ]:
#Plot confusion Matrix for Decision Tree
y_pred_test_dt_params = pipeline_dt_params.predict(X_test)
conf_matrix_dt_params = confusion_matrix(y_test, y_pred_test_dt_params)
print("Confusion Matrix:")
print(conf_matrix_dt_params)

cm_display_dt_params = ConfusionMatrixDisplay(confusion_matrix = conf_matrix_dt_params, display_labels = ["No Delay", "Delay"])
# display matrix
cm_display_dt_params.plot()
plt.title('Decision Tree')
plt.show()

In [ ]:
#Plot confusion Matrix for Logistic Regression
y_pred_test_sgdlr = pipeline_sgdlr.predict(X_test)
conf_matrix_sgdlr = confusion_matrix(y_test, y_pred_test_sgdlr)
print("Confusion Matrix:")
print(conf_matrix_sgdlr)

cm_display_sgdlr = ConfusionMatrixDisplay(confusion_matrix = conf_matrix_sgdlr, display_labels = ["No Delay", "Delay"])
# display matrix
cm_display_sgdlr.plot()
plt.title('Logistic Regression')
plt.show()

In [ ]:
#Plot confusion Matrix for Gradient Boosting
y_pred_test_gb = pipeline_gb.predict(X_test)
conf_matrix_gb= confusion_matrix(y_test, y_pred_test_gb)
print("Confusion Matrix:")
print(conf_matrix_gb)

cm_display_gb = ConfusionMatrixDisplay(confusion_matrix = conf_matrix_gb, display_labels = ["No Delay", "Delay"])
# display matrix
cm_display_gb.plot()
plt.title('Gradient Boosting')
plt.show()

After comparing the models Logistic Regression performs best overall because it has the best recall for delays, while Gradient Boosting comes second with more delays missed, and last is Decision Tree which has a higher missed delays count. So for our problem statement in which is predicting flight delays, it is best to warn too often than to miss real delays, which Logistic Regression performs best at. It also higlights the dificulty of predicting airline delays as there are too many factors to consider and it is a very volatile area.